In [ ]:
"""
将 USalign 的输出转换为易读的对齐摘要，并结合模板基序给出自然语言说明。
"""

from __future__ import annotations

from typing import Any, Dict, List, Optional, Tuple
import subprocess

from eval_utils import DEFAULT_USALIGN_BIN


def run_usalign(
    gt_path: str,
    pred_path: str,
    usalign_bin: str = DEFAULT_USALIGN_BIN,
    return_output: bool = False,
) -> Tuple[Optional[float], Optional[float]] | Tuple[Optional[float], Optional[float], List[str]]:
    """
    调用 USalign（不使用 outfmt），解析默认文本输出中的 TM-score / RMSD。
    return_output=True 时额外返回 stdout 行列表。
    """
    try:
        completed = subprocess.run(
            [usalign_bin, gt_path, pred_path, '-TMscore', '2', '-atom', " C1'", '-ter', '1'],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
            text=True,
        )
        lines = completed.stdout.strip().splitlines()

        tm = None
        rmsd = None

        # 典型行示例: "User-specified initial alignment: TM/Lali/rmsd = 0.28023,   67, 10.611"
        for line in lines:
            if 'TM/Lali/rmsd' in line:
                try:
                    after_eq = line.split('=', 1)[1]
                    tokens = after_eq.replace(',', ' ').split()
                    numbers = []
                    for tok in tokens:
                        try:
                            numbers.append(float(tok))
                        except ValueError:
                            continue
                    if len(numbers) >= 3:
                        tm = numbers[0]
                        rmsd = numbers[2]
                except Exception:
                    tm = None
                break

        if return_output:
            return tm, rmsd, lines
        return tm, rmsd
    except subprocess.CalledProcessError:
        if return_output:
            return None, None, []
        return None, None


def _extract_alignment_block(lines: List[str]) -> List[str]:
    """
    从 USalign 默认输出中截取倒数第 6~4 行的三行对齐片段。
    若行数不足，则返回空列表。
    """

    block = lines[-4].strip()
    return block


def _describe_motif_alignment(alignment_block: List[str], motif_template: str) -> List[str]:
    """
    根据 motif 模板（如 G----UUCACGU...）逐位描述是否对齐。
    alignment_block 期望为 [seq1, markers, seq2]，markers 中 ':' 表示 <5Å。
    """
    if len(alignment_block) < 3:
        return []

    markers = alignment_block
    descriptions: List[str] = []

    for idx, base in enumerate(motif_template):
        if base in ("-", " "):
            continue
        marker = markers[idx] if idx < len(markers) else " "
        if marker == ":":
            status = "<5Å 对齐"
        elif marker == ".":
            status = "已对齐但距离>5Å或错配"
        else:
            status = "未对齐/缺口"
        descriptions.append(f"位置 {idx+1} motif碱基 {base}: {status}")

    return descriptions


def usalign_to_nl(
    gt_path: str,
    pred_path: str,
    motif_template: str = "",
    usalign_bin: str = DEFAULT_USALIGN_BIN,
) -> Dict[str, Any]:
    """
    运行 USalign（无 outfmt 选项），提取：
    - TM score / RMSD
    - 倒数第 6~4 行的三行字符对齐
    - 基于 motif 模板的对齐描述（仅对非 '-' 位置）
    """
    tm, rmsd, lines = run_usalign(gt_path, pred_path, usalign_bin=usalign_bin, return_output=True)
    alignment_block = _extract_alignment_block(lines)
    motif_desc = _describe_motif_alignment(alignment_block, motif_template) if motif_template else []

    return {
        "gt_path": gt_path,
        "pred_path": pred_path,
        "tm_score": tm,
        "rmsd": rmsd,
        "alignment_block": alignment_block,  # 典型为三行：seq1 / 标记 / seq2
        "motif_template": motif_template,
        "motif_alignment_desc": motif_desc,
        "raw_output_lines": lines,
    }


In [11]:

motif='-----------UUACAA---------------G-GUA--------------------UAA-------------'
data  = usalign_to_nl('/workspace/hedongchen/Qwen-Agent/tests/pipeline_test_output_twisterv2/01_backbone/gen_len73_1000samples/ss_conditional_sample_8.pdb','/workspace/hedongchen/Qwen-Agent/tests/pipeline_test_output_twisterv2/04_structure_pred/ss_conditional_sample_8_0/ss_conditional_sample_8_0_model.cif',motif)

In [12]:
data

{'gt_path': '/workspace/hedongchen/Qwen-Agent/tests/pipeline_test_output_twisterv2/01_backbone/gen_len73_1000samples/ss_conditional_sample_8.pdb',
 'pred_path': '/workspace/hedongchen/Qwen-Agent/tests/pipeline_test_output_twisterv2/04_structure_pred/ss_conditional_sample_8_0/ss_conditional_sample_8_0_model.cif',
 'tm_score': 0.28522,
 'rmsd': 15.191,
 'alignment_block': '....................................::::::::::::::::::::::...............',
 'motif_template': '-----------UUACAA---------------G-GUA--------------------UAA-------------',
 'motif_alignment_desc': ['位置 12 motif碱基 U: 已对齐但距离>5Å或错配',
  '位置 13 motif碱基 U: 已对齐但距离>5Å或错配',
  '位置 14 motif碱基 A: 已对齐但距离>5Å或错配',
  '位置 15 motif碱基 C: 已对齐但距离>5Å或错配',
  '位置 16 motif碱基 A: 已对齐但距离>5Å或错配',
  '位置 17 motif碱基 A: 已对齐但距离>5Å或错配',
  '位置 33 motif碱基 G: 已对齐但距离>5Å或错配',
  '位置 35 motif碱基 G: 已对齐但距离>5Å或错配',
  '位置 36 motif碱基 U: 已对齐但距离>5Å或错配',
  '位置 37 motif碱基 A: <5Å 对齐',
  '位置 58 motif碱基 U: <5Å 对齐',
  '位置 59 motif碱基 A: 已对齐但距离>5Å或错配',
  '位置 60 motif碱基 A: 已对